# Importação das Bibliotecas

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import cv2
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import os

import albumentations as A

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import  Input, Conv2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.utils import Sequence
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import keras.layers as layers
import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.applications import ResNet50

from sklearn.metrics import accuracy_score, precision_score, recall_score

from sklearn.metrics import roc_curve, auc

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# Importação do Dataset

In [ ]:
#Caminho das pastas

main_path = '256x256/'

train_dataframe = pd.read_csv(os.path.join(main_path, 'train_split.csv'))

val_dataframe = pd.read_csv(os.path.join(main_path, 'val_split.csv'))

In [ ]:
train_dataframe

In [ ]:
val_dataframe


In [ ]:
intersecao = set(train_dataframe['patient_id']).intersection(set(val_dataframe['patient_id']))
print(f"Pacientes em comum: {len(intersecao)}")

# Peso das classes

In [ ]:
Num_classes = 2

Num_saudaveis = train_dataframe['label'].value_counts()[0] + val_dataframe['label'].value_counts()[0]

#print(Num_saudaveis)

Num_fraturados = train_dataframe['label'].value_counts()[1] + val_dataframe['label'].value_counts()[1]

#print(Num_fraturados)

weight_saudaveis = (Num_saudaveis + Num_fraturados) / (Num_classes * Num_saudaveis)

print(f"Pesos dos slices saudáveis: {round(weight_saudaveis, 5)} \n")

weight_fraturados = (Num_saudaveis + Num_fraturados) / (Num_classes * Num_fraturados)

print(f"Pesos dos slices fraturados: {round(weight_fraturados, 5)} \n")

# Carregamento e Normalização das imagens

In [ ]:
def load_image(filepath):
    
    try:
        
        img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        
        if img is None: 
            return None
        
    
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        return img_rgb
    
    except Exception as e:
        print(f"Erro ao carregar {filepath}: {e}")
        return None

normalization_resnet50 = A.Normalize( mean = (0.485, 0.456, 0.406),
                                      std =(0.229, 0.224, 0.225)
                                    ) 

# Data Augmentation

In [ ]:
train_DataAugmentation = A.Compose([

    A.HorizontalFlip(p=0.5),

    A.Rotate(limit = (-20,20), interpolation = cv2.INTER_LINEAR, border_mode = cv2.BORDER_CONSTANT, fill = 0, p = 0.6), #random rotation

    A.RandomScale(scale_limit = (-0.12,0.12), interpolation = cv2.INTER_LINEAR, mask_interpolation = cv2.INTER_LINEAR, p = 0.3), #zoom, depois testar com p=0.75

    A.ElasticTransform(alpha=350.0, sigma=20, interpolation=cv2.INTER_CUBIC, p=0.5, border_mode=cv2.BORDER_CONSTANT, fill = 0),

    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.14, brightness_by_max = False, ensure_safe_range = False, p=0.2),

    A.GaussianBlur(sigma_limit=(0.2, 0.7), blur_limit = (3,3), p=0.7),

    A.PadIfNeeded(min_height=256, min_width=256, border_mode=cv2.BORDER_CONSTANT, fill=0),
    A.CenterCrop(height=256, width=256)
])

#train_DataAugmentation = ImageDataGenerator(
#    rotation_range = 45,
#    horizontal_flip = True    
#)

#Ajustar o data augmentation pra deixar conforme as referências: random horizontal flip, 
#random rotation, Gaussian blur, Translation, Zoomed-in/scaling, Elastic transformation,
#Rotation e Contrast/brightness enhancement.

#Cervical Spine Fracture Detection and Classification Using Two-Stage Deep Learning Methodology
#Improving Vertebral Fracture Detection in C-Spine CT Images Using Bayesian Probability-Based Ensemble Learning 

# Data Generators

In [ ]:
class_weights = {
    0: weight_saudaveis,
    1: weight_fraturados
}

In [ ]:
#https://mahmoudyusof.github.io/blog/
#https://medium.com/@lucas.lvtj/explorando-a-classifica%C3%A7%C3%A3o-de-imagens-com-resnet50-19ee263393e3

class DataGenerator(Sequence):
    
    def __init__(self, dataframe, batch_size = 4, shuffle = False, augmentation = None, **kwargs):
        super().__init__(**kwargs)
        self.df = dataframe
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_classes = 2
        self.augmentation = augmentation
        self.on_epoch_end()
        
    def __getitem__(self, idx):
        indices = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        df_batch = self.df.iloc[indices]
        X, y = self.__data_generation(df_batch)
        return X, y
        
    
    def on_epoch_end(self):
        self.indices = np.arange(len(self.df))
        if self.shuffle == True:
            np.random.shuffle(self.indices)
    
    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))
    
       
    def __data_generation(self, df_batch):
        
        current_batch_size = len(df_batch)
        X = np.zeros((self.batch_size, 256, 256, 3), dtype = np.float32)
        y = np.zeros((self.batch_size), dtype = np.uint8)
        
        for i, (idx, row) in enumerate(df_batch.iterrows()):
            
           image = load_image(row['filepath'])
           
           if image is not None:
                          
               if self.augmentation is not None:
                                                         
                   augmented = self.augmentation(image = image)
                   image = augmented['image']
                
               img_normalized_dict = normalization_resnet50(image = image)
               image_final = img_normalized_dict['image']
                
               X[i] = image_final
               y[i] = row['label']
                
           else: 
               
               print(f"erro{row['filepath']}")
               y[i] = row['label']        
             
        y_one_hot = tf.keras.utils.to_categorical(y, num_classes = self.num_classes)
        
        return X, y_one_hot  

In [ ]:
Batch_size = 16

train_generator = DataGenerator(train_dataframe, batch_size= Batch_size, shuffle = True, augmentation = train_DataAugmentation)

val_generator = DataGenerator(val_dataframe, batch_size= Batch_size, shuffle = False, augmentation = None)

# Arquitetura ResNet50

In [ ]:

#Normalizar as imagens de acordo com a normalização do modelo ResNet50

def model_ResNet50_Transfer(input_shape=(256, 256, 3), num_classes=2, trainable_base=False):

    inputs = Input(shape=input_shape)
    
    resnet_base = ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)

    resnet_base.trainable = trainable_base

    x = layers.GlobalMaxPooling2D()(resnet_base.output)

    x = Dense(512, activation='relu')(x) #kernel_regularizer=tf.keras.regularizers.l2(0.0001)
    x = Dropout(0.5)(x)#Testar com 0.7

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    predictions = Dense(num_classes, activation=None)(x)

    model = Model(inputs=inputs, outputs=predictions)

    return model, resnet_base



In [ ]:
model, resnet_base = model_ResNet50_Transfer(trainable_base=False)

model.compile(optimizer=AdamW(learning_rate= 1e-4,
                              weight_decay = 1e-3), 
              loss=CategoricalCrossentropy(from_logits = True), 
              metrics=['accuracy'])

print("Model fit 1 - Pesos congelados")
model.fit(train_generator, epochs=10, validation_data=val_generator, class_weight=class_weights)

#Alterar learning_rate e/ou número de épocas.

In [ ]:
early_stopping = EarlyStopping(monitor = 'val_loss', patience = 12, restore_best_weights = True)

reduce_plateau = ReduceLROnPlateau(monitor = 'val_loss', factor = 0.5, patience = 6, min_lr = 1e-7, verbose = 1)

In [ ]:
resnet_base.trainable = True

model.compile(optimizer=AdamW(learning_rate= 1e-6,
                              weight_decay = 1e-3),
              loss=CategoricalCrossentropy(from_logits = True),
              metrics=['accuracy'],
              jit_compile = False)

#Alterar learning_rate e/ou número de épocas.

history = model.fit(train_generator,
                    epochs=75,
                    validation_data=val_generator,
                    callbacks=[early_stopping, reduce_plateau],
                    class_weight=class_weights)

#Matriz de Confusão e Gráficos

In [ ]:
predict_logits = model.predict(val_generator)

predict_res = tf.nn.softmax(predict_logits).numpy()

predict_labels = np.argmax(predict_res, axis = 1)

y_true = []

for i in range(len(val_generator)):
    
    _, batch_y_one_hot = val_generator[i]
    batch_y_labels = np.argmax(batch_y_one_hot, axis = 1)
    y_true.extend(batch_y_labels)

y_true = np.array(y_true)

acc = accuracy_score(y_true , predict_labels)

recall = recall_score(y_true, predict_labels)

precision = precision_score(y_true, predict_labels)

f1 = f1_score(y_true, predict_labels)


cm = confusion_matrix(y_true, predict_labels)
cm_normalizada = cm.astype('float') / cm.sum(axis = 1)[:, np.newaxis]

labels = [f"{v}\n({p:.2%})" for v, p in zip(cm.flatten(), cm_normalizada.flatten())]
labels = np.asarray(labels).reshape(cm.shape)

tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp)

plt.figure(figsize=(10,8))

sns.heatmap(cm, annot = True, fmt ='d', cmap = 'Blues',
            xticklabels = ['Saudável', 'Fraturado'],
            yticklabels = ['Saudável', 'Fraturado'],
            annot_kws={"size":15})

plt.title('Matriz de Confusão - Resnet50')
plt.ylabel('Rótulo Verdadeiro')
plt.xlabel('Rótulo Previsto')
plt.show()


print(f"Acurácia:{round(acc,5)} \n Recall: {round(recall,5)} \n Precisão: {round(precision,5)} \n F1_Score: {round(f1,5)} \n Especificidade: {round(specificity,5)} \n")


print('Relatório')
print(classification_report(y_true, predict_labels, target_names = ['Saudável','Fraturado']))

In [ ]:


def plot_roc_curve(y_true, y_probs):
    
    fpr, tpr, thresholds = roc_curve(y_true, y_probs[:, 1])
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (área = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Linha de base (aleatória)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taxa de Falsos Positivos (1 - Especificidade)')
    plt.ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)')
    plt.title('Curva ROC - Detecção de Fraturas - ResNet50')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
def plot_learning_curves(history):
    
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))

    # Gráfico de Acurácia
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, val_acc, label='Validação')
    plt.title('Acurácia de Treino e Validação - ResNet50')
    plt.xlabel('Épocas')
    plt.ylabel('Acurácia')
    plt.legend(loc='lower right')
    plt.grid(alpha = 0.3)

    # Gráfico de Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Treino')
    plt.plot(epochs_range, val_loss, label='Validação')
    plt.title('Treino e Validação - Loss - ResNet50')
    plt.xlabel('Épocas')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(alpha = 0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_learning_curves(history)

In [ ]:
predict_logits = model.predict(val_generator)
predict_probs = tf.nn.softmax(predict_logits).numpy()

plot_roc_curve(y_true, predict_res)